In [ ]:
import cupy as cp
import numpy as np
from time import perf_counter
import matplotlib.pyplot as plt

In [ ]:
# parameters
size = 90000
maxit = 30000

In [ ]:
# init arrays on cpu
#value_grid_real = cp.zeros([size, size], dtype=cp.float32)
#value_grid_imag = cp.zeros([size, size], dtype=cp.float32)
#value_grid_mask = cp.zeros([size, size], dtype=cp.int16)
#coordinate_grid_real = cp.zeros([size, size], dtype=cp.float32)
#coordinate_grid_imag = cp.zeros([size, size], dtype=cp.float32)

value_grid_mask = cp.zeros(size * size, dtype=cp.int16)

In [ ]:
# fill coordinate arrays
for i in range(size):
    coordinate_grid_real[:, i] = 4 * (i - size / 2) / size
    coordinate_grid_imag[i, :] = 4 * (i - size / 2) / size

In [ ]:
# old skipmask
a = coordinate_grid_real - 0.25
b = coordinate_grid_imag
c = 0.25
r2 = a**2 + b**2

skip_mask = (
    ((a + 1.25)**2 + b**2 < 0.0625)
    | (r2**2 + 4*c*a*r2 - 4*c**2*b**2 < 0)            # main cardioid
    | ((a + 1.558)**2 + b**2 < 0.003)
    | ((a + 0.375)**2 + (b + 0.745)**2 < 0.008)
    | ((a + 0.375)**2 + (b - 0.745)**2 < 0.008)
)

value_grid_mask[skip_mask] = maxit

In [ ]:
# debug
#plt.imshow(coordinate_grid_real, origin='lower')
#plt.colorbar()
#plt.show()

#plt.imshow(coordinate_grid_imag, origin='lower')
#plt.colorbar()
#plt.show()

#plt.imshow(cp.asnumpy(value_grid_mask), origin='lower')
#plt.colorbar()
#plt.show()


In [ ]:
#v1
start = perf_counter()

it = 0
while it < maxit:
    it += 1
    # square value_grid
    # (a + bi)^2 = a^2 - b^2 + 2abi
    value_grid_temp = value_grid_real**2 - value_grid_imag**2
    value_grid_imag = 2 * value_grid_imag * value_grid_real
    value_grid_real = value_grid_temp
    
    # add coordinate_grid to value_grid 
    # (a + bi) + (c + di) = (a + c) + (b + d)i
    value_grid_real = value_grid_real + coordinate_grid_real
    value_grid_imag = value_grid_imag + coordinate_grid_imag
    
    # iteration mask
    # mask[(|value| > 4)] = it
    value_grid_mask[(value_grid_real**2 + value_grid_imag**2) > 4] = it

# set everything that hasn't escaped to max iterations
value_grid_mask[value_grid_mask == 0] = maxit

print(f"This took {round(perf_counter() - start, 6)} seconds")

plt.imshow(cp.asnumpy(value_grid_mask))
plt.colorbar()
plt.show()

In [ ]:
#v2
start = perf_counter()

# set active mask to all values in the grid_mask that are 0
active = (value_grid_mask < maxit)

it = 0
while it < maxit:
    it += 1

    # only apply calculations to un-masked points
    value_real = value_grid_real[active]
    value_imag = value_grid_imag[active]
    coord_real = coordinate_grid_real[active]
    coord_imag = coordinate_grid_imag[active]
    
    # square value_grid
    # (a + bi)^2 = a^2 - b^2 + 2abi
    value_temp = value_real**2 - value_imag**2
    value_imag = 2 * value_imag * value_real
    value_real = value_temp
    
    # add coordinate_grid to value_grid 
    # (a + bi) + (c + di) = (a + c) + (b + d)i
    value_real = value_real + coord_real
    value_imag = value_imag + coord_imag

    # apply to grids
    value_grid_real[active] = value_real
    value_grid_imag[active] = value_imag
    
    # iteration mask
    # mask[active & (|value| > 4)] = it
    escaped = active & ((value_grid_real**2 + value_grid_imag**2) > 4)
    value_grid_mask[escaped] = it
    active &= ~escaped

    #if not active.any(): break
    
    #fig, (ax1, ax2) = plt.subplots(1, 2)
    #ax1.imshow(np.abs(value_grid_real) < 2)
    #ax2.imshow(np.abs(value_grid_imag) < 2)
    #plt.show()
    
    #plt.imshow(it * cp.asnumpy(active))
    #plt.colorbar()
    #plt.show()

# set everything that hasn't escaped to max iterations
value_grid_mask[value_grid_mask == 0] = maxit

print(f"This took {round(perf_counter() - start, 6)} seconds")

plt.imshow(cp.asnumpy(value_grid_mask))
plt.colorbar()
plt.show()

In [ ]:
#v3
start = perf_counter()

active = ~skip_mask
value_grid_mask[skip_mask] = maxit

it = 0
while it < maxit:
    it += 1

    new_real = value_grid_real**2 - value_grid_imag**2 + coordinate_grid_real
    new_imag = 2 * value_grid_real * value_grid_imag + coordinate_grid_imag

    value_grid_real = cp.where(active, new_real, value_grid_real)
    value_grid_imag = cp.where(active, new_imag, value_grid_imag)

    escaped = active & ((value_grid_real**2 + value_grid_imag**2) > 4)
    value_grid_mask[escaped] = it
    active &= ~escaped

# set everything that hasn't escaped to max iterations
value_grid_mask[value_grid_mask == 0] = maxit

print(f"This took {round(perf_counter() - start, 6)} seconds")

plt.imshow(cp.asnumpy(value_grid_mask))
plt.colorbar()
plt.show()

In [ ]:
#v4
start = perf_counter()

mandelbrot_kernel = cp.ElementwiseKernel(
    'float32 cr, float32 ci, int32 maxit, bool skip',  # inputs
    'float32 vr, float32 vi, int16 mask',              # outputs
    '''
    if (skip) {
        vr = 0; vi = 0; mask = maxit;
    } else {
        float zr = 0, zi = 0, tmp;
        int it = 0;
        while (it < maxit) {
            it++;
            tmp = zr*zr - zi*zi + cr;
            zi  = 2*zr*zi + ci;
            zr  = tmp;
            if (zr*zr + zi*zi > 4.0f) { mask = it; zr = 0; zi = 0; break; }
        }
        if (mask == 0) mask = maxit;
        vr = zr; vi = zi;
    }
    ''',
    'mandelbrot'
)

mandelbrot_kernel(
    coordinate_grid_real, coordinate_grid_imag,
    maxit, skip_mask,
    value_grid_real, value_grid_imag, value_grid_mask
)

print(f"This took {round(perf_counter() - start, 6)} seconds")

plt.imshow(cp.asnumpy(value_grid_mask))
plt.colorbar()
plt.show()


In [ ]:
#v5
start = perf_counter()

mandelbrot_kernel = cp.ElementwiseKernel(
    'float32 cr, float32 ci, int32 maxit',
    'float32 vr, float32 vi, int16 mask',
    '''
    double a  = cr - 0.25;
    double r2 = a*a + ci*ci;
    bool skip = (
        ((cr + 1.0)*(cr + 1.0)   + ci*ci) < 0.0625
        || (r2*r2 + a*r2 - 0.0625*ci*ci)  < 0
        || ((cr + 1.308)*(cr + 1.308)     + ci*ci) < 0.003
        || ((cr + 0.125)*(cr + 0.125)     + (ci + 0.745)*(ci + 0.745)) < 0.008
        || ((cr + 0.125)*(cr + 0.125)     + (ci - 0.745)*(ci - 0.745)) < 0.008
    );

    if (skip) {
        vr = 0; vi = 0; mask = maxit;
    } else {
        double zr = 0, zi = 0, tmp;
        int it = 0;
        while (it < maxit) {
            it++;
            tmp = zr*zr - zi*zi + cr;
            zi  = 2*zr*zi + ci;
            zr  = tmp;
            if (zr*zr + zi*zi > 4.0) { mask = it; zr = 0; zi = 0; break; }
        }
        if (mask == 0) mask = maxit;
        vr = zr; vi = zi;
    }
    ''',
    'mandelbrot'
)

# call with no skip_mask
mandelbrot_kernel(
    coordinate_grid_real, coordinate_grid_imag, maxit,
    value_grid_real, value_grid_imag, value_grid_mask
)

cp.cuda.Stream.null.synchronize()  # wait for gpu

print(f"This took {round(perf_counter() - start, 6)} seconds")

plt.imshow(cp.asnumpy(value_grid_mask))
plt.colorbar()
plt.show()

In [ ]:
#v6
mandelbrot_kernel = cp.ElementwiseKernel(
    'int32 size, int32 maxit',
    'int16 mask',
    '''
    int ix = i % size;
    int iy = i / size;
    double cr = 4.0 * (ix - size * 0.5) / size;
    double ci = 4.0 * (iy - size * 0.5) / size;

    // skip internal points
    double a  = cr - 0.25;
    double r2 = a*a + ci*ci;
    bool skip = (
        ((cr + 1.0)*(cr + 1.0)  + ci*ci) < 0.0625
        || (r2*r2 + a*r2 - 0.0625*ci*ci) < 0
        || ((cr + 1.308)*(cr + 1.308)    + ci*ci) < 0.003
        || ((cr + 0.125)*(cr + 0.125)    + (ci + 0.745)*(ci + 0.745)) < 0.008
        || ((cr + 0.125)*(cr + 0.125)    + (ci - 0.745)*(ci - 0.745)) < 0.008
    );

    if (skip) {
        mask = maxit;
    } else {
        double zr = 0, zi = 0, tmp;
        int it = 0;
        while (it < maxit) {
            it++;
            tmp = zr*zr - zi*zi + cr;
            zi  = 2*zr*zi + ci;
            zr  = tmp;
            if (zr*zr + zi*zi > 4.0) { mask = it; break; }
        }
        if (mask == 0) mask = maxit;
    }
    ''',
    'mandelbrot'
)

start = perf_counter()
mandelbrot_kernel(cp.int32(size), cp.int32(maxit), value_grid_mask)
cp.cuda.Stream.null.synchronize() # wait for the gpu

print(f"This took {round(perf_counter() - start, 6)} seconds")

plt.imshow(cp.asnumpy(value_grid_mask).reshape(size, size))
plt.colorbar()
plt.show()

In [ ]:
#v7
mandelbrot_kernel = cp.ElementwiseKernel(
    'int32 size, int32 maxit', # input: image size, max iterations
    'int16 mask',              # output: array containing iteration count
    '''
    // i is the pixel index, provided by CuPy
    // conversion from pixel index to space coordinate
    int ix = i % size;
    int iy = i / size;
    double cr = 4.0 * (ix - size * 0.5) / size;
    double ci = 4.0 * (iy - size * 0.5) / size;

    // main bulb and interior region skips; points here won't ever escape
    double a  = cr - 0.25;
    double r2 = a*a + ci*ci;
    bool skip = (
        ((cr + 1.0)*(cr + 1.0)  + ci*ci) < 0.0625
        || (r2*r2 + a*r2 - 0.0625*ci*ci) < 0
        || ((cr + 1.308)*(cr + 1.308)    + ci*ci) < 0.003
        || ((cr + 0.125)*(cr + 0.125)    + (ci + 0.745)*(ci + 0.745)) < 0.008
        || ((cr + 0.125)*(cr + 0.125)    + (ci - 0.745)*(ci - 0.745)) < 0.008
    );

    if (skip) {
        mask = maxit;
    } else {
        // MAIN LOOP
        double zr = 0, zi = 0, tmp;      // real, imag, tmp
        double zr_old = 0, zi_old = 0;   // old_real, old_imag for periodicity checks
        double zr2, zi2;                 // real^2, imag^2
        int it = 0, check_every = 20;    // current it; length of periodicity check
        while (it < maxit) {
            it++;

            // compute z --> z^2 + c
            zr2 = zr * zr;
            zi2 = zi * zi;
            tmp = zr2 - zi2 + cr;
            zi  = fma(2.0 * zr, zi, ci); // fancy Fused-Multiply-Add
            zr  = tmp;

            // if out of bounds, break
            if (zr2 + zi2 > 4.0) { mask = it; break; }

            // if periodic, break
            if (zr == zr_old && zi == zi_old) { mask = maxit; break; }
            if (it % check_every == 0) { zr_old = zr; zi_old = zi; }
        }
        if (mask == 0) mask = maxit;
    }
    ''',
    'mandelbrot'
)
start = perf_counter()
mandelbrot_kernel(cp.int32(size), cp.int32(maxit), value_grid_mask)
cp.cuda.Stream.null.synchronize() # wait for the gpu

print(f"This took {round(perf_counter() - start, 6)} seconds")

plt.imshow(cp.asnumpy(value_grid_mask).reshape(size, size))
plt.colorbar()
plt.show()

In [ ]:
#v8
mandelbrot_kernel = cp.ElementwiseKernel(
    'int32 size, int32 maxit', # input: image size, max iterations
    'int16 mask',              # output: array containing iteration count
    '''
    // i is the pixel index, provided by CuPy
    // conversion from pixel index to space coordinate
    int ix = i % size;
    int iy = i / size;
    double cr = 4.0 * (ix - size * 0.5) / size;
    double ci = 4.0 * (iy - size * 0.5) / size;

    // main bulb and interior region skips; points here won't ever escape
    double a  = cr - 0.25;
    double r2 = a*a + ci*ci;
    bool skip = (
        ((cr + 1.0)*(cr + 1.0)  + ci*ci) < 0.0625
        || (r2*r2 + a*r2 - 0.0625*ci*ci) < 0
        || ((cr + 1.308)*(cr + 1.308)    + ci*ci) < 0.003
        || ((cr + 0.125)*(cr + 0.125)    + (ci + 0.745)*(ci + 0.745)) < 0.008
        || ((cr + 0.125)*(cr + 0.125)    + (ci - 0.745)*(ci - 0.745)) < 0.008
    );

    if (skip) {
        mask = maxit;
    } else {
        // MAIN LOOP
        double zr = 0, zi = 0, tmp;     // real, imag, tmp
        double zr2, zi2;                // old_real, old_imag for periodicity checks
        double zr_old = 0, zi_old = 0;  // real^2, imag^2
        int it = 0;                     // current iteration
        int period = 1;                 // current cycle length being tested
        int countdown = 1;              // iterations until next reference update
        
        while (it < maxit) {
            it++;

            // compute z --> z^2 + c
            zr2 = zr * zr;
            zi2 = zi * zi;
            tmp = zr2 - zi2 + cr;
            zi  = fma(2.0 * zr, zi, ci); // fancy Fused-Multiply-Add
            zr  = tmp;
            
            // if out of bounds, break
            if (zr2 + zi2 > 4.0) { mask = it; break; }

            // if periodic, break
            if (zr == zr_old && zi == zi_old) { mask = maxit; break; }

            // Brent's Cycle detection algorithm
            countdown--;
            if (countdown == 0) {
                zr_old    = zr;
                zi_old    = zi;
                period    *= 2;      // double the window each time
                countdown = period;  // reset countdown to new period
            }
        }
        if (mask == 0) mask = maxit;
    }
    ''',
    'mandelbrot'
)
start = perf_counter()
mandelbrot_kernel(cp.int32(size), cp.int32(maxit), value_grid_mask)
cp.cuda.Stream.null.synchronize() # wait for the gpu

print(f"This took {round(perf_counter() - start, 6)} seconds")

plt.imshow(cp.asnumpy(value_grid_mask).reshape(size, size))
plt.colorbar()
plt.show()